In [19]:
import polars as pl
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl import Workbook,utils
from openpyxl.drawing.image import Image
from procyclingstats import Race, Rider,Team

carrera="race/itzulia-basque-country/2026"
race_name = carrera.split("/")[1]

team = Team("/team/burgos-burpellet-bh-2026")
riders = pl.DataFrame(team.riders())

# Acumular datos
data = []
print(f"Buscando participaciones de los corredores de {team.name()} en {carrera}...´{race_name}")
for rider in riders.iter_rows(named=True):
    rider_url = rider["rider_url"]
    if not rider_url.startswith("/"):
        rider_url = f"/{rider_url}"
    r = Rider(rider_url)
    rider_name = r.name()
    seasons = pl.DataFrame(r.teams_history())["season"].to_list()
    
    c=0
    for season in seasons[:10]:
        
        res = Rider(rider_url + f"/{season}")
        participations = pl.DataFrame(res.season_results())
        
        if len(participations) > 0:
            # Filtrar por carrera buscando en stage_url
                        
            race_participations = participations.filter(
                pl.col("stage_url").str.contains(race_name)
            )
            
            if race_participations.height > 0:
                if c==0:print(f"Revisando {rider_name}")
                c+=1    
                print(f"Año {season}...")
                for row in race_participations.iter_rows(named=True):
                    result=row["result"] if row["result"] is not None else "DNF"
                    data.append({
                        "rider_name": rider_name,
                        "season": season,
                        
                        "stage_name": row["stage_name"],
                        "result": result,
                        "uci_points": row["uci_points"]
                    })
                    print(f"{row['stage_name']}: {result}: {row['uci_points']} puntos")

# Convertir a DataFrame y agrupar
if data:
    result_df = pl.DataFrame(data)
    
else:
    print("No se encontraron participaciones")

Buscando participaciones de los corredores de Burgos Burpellet BH en race/itzulia-basque-country/2026...´itzulia-basque-country
Revisando Jesús Herrada
Año 2021...
Mountains classificationMountains classification: 26: 0.0 puntos
General classificationGeneral classification: 46: 8.0 puntos
S6Stage 6 - Ondarroa › Arrate (Eibar): 50: 0.0 puntos
S5Stage 5 - Hondarribia › Ondarroa: 93: 0.0 puntos
S4Stage 4 - Vitoria-Gasteiz › Hondarribia: 97: 0.0 puntos
S3Stage 3 - Amurrio › Ermualde (Laudio): 42: 0.0 puntos
S2Stage 2 - Zalla › Sestao: 45: 0.0 puntos
S1 (ITT)Stage 1 (ITT) - Bilbao › Bilbao: 52: 0.0 puntos
Año 2019...
Mountains classificationMountains classification: 26: 0.0 puntos
General classificationGeneral classification: 66: 0.0 puntos
S6Stage 6 - Eibar › Eibar: 43: 0.0 puntos
S5Stage 5 - Arrigorriaga › Arrate: 67: 0.0 puntos
S4Stage 4 - Vitoria-Gasteiz › Arrigorriaga: 101: 0.0 puntos
S3Stage 3 - Sarriguren › Estibaliz: 26: 0.0 puntos
S2Stage 2 - Zumarraga › Gorraiz: 117: 0.0 puntos
S1

In [20]:
# Resumen agrupado por ciclista
if data:
    summary = result_df.group_by("rider_name").agg([
        pl.col("rider_name").first().alias("Ciclista"),
        pl.col("season").n_unique().alias("Participaciones"),        
        pl.col("result").min().alias("Mejor Resultado"),
        pl.col("result").max().alias("Peor Resultado"),
        pl.col("uci_points").max().round(0).cast(pl.Int64, strict=False).alias("MAX puntos UCI")
    ]).sort("Participaciones")
    
    print("Resumen por ciclista (ordenado por participaciones):")
    print(summary)

    grouped_df = result_df.group_by(["rider_name", "season"]).agg([
        pl.len().alias("num_registros"),
        pl.col("uci_points").sum().round(0).cast(pl.Int64, strict=False).alias("puntos_uci_totales"),
        pl.col("result").sort().alias("resultados"),
        pl.col("stage_name").sort().alias("etapas")
    ]).sort(["rider_name", "season"])

    print("\nresult_df agrupado por rider_name y season:")
    print(grouped_df)

Resumen por ciclista (ordenado por participaciones):
shape: (10, 6)
┌─────────────────┬─────────────────┬────────────────┬───────────┬────────────────┬────────────────┐
│ rider_name      ┆ Ciclista        ┆ Participacione ┆ Mejor     ┆ Peor Resultado ┆ MAX puntos UCI │
│ ---             ┆ ---             ┆ s              ┆ Resultado ┆ ---            ┆ ---            │
│ str             ┆ str             ┆ ---            ┆ ---       ┆ str            ┆ i64            │
│                 ┆                 ┆ u32            ┆ str       ┆                ┆                │
╞═════════════════╪═════════════════╪════════════════╪═══════════╪════════════════╪════════════════╡
│ Jambaljamts     ┆ Jambaljamts     ┆ 1              ┆ 125       ┆ DNF            ┆ 0              │
│ Sainbayar       ┆ Sainbayar       ┆                ┆           ┆                ┆                │
│ Sinuhé          ┆ Sinuhé          ┆ 1              ┆ 158       ┆ 83             ┆ 2              │
│ Fernández       ┆ Fer